<a href="https://colab.research.google.com/github/FCENA-PUCE/MetodosNumericos-05-N0062/blob/main/2-Notebooks/03-Sistemas-de-ecuaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="border: none; border-collapse: collapse;">
    <tr>
        <td style="width: 20%; vertical-align: middle; padding-right: 10px;">
            <img src="https://i.imgur.com/X5m8yDo.png" width="120">
        </td>
        <td style="width: 2px; text-align: center;">
            <font color="#1957d1" size="7">|</font><br>
            <font color="#1957d1" size="7">|</font>
        </td>
        <td>
            <p style="font-variant: small-caps;"><font color="#1957d1" size="5">
                <b>Facultad de Ciencias Exactas, Naturales y Ambientales</b>
            </font> </p>
            <p style="font-variant: small-caps;"><font color="#1957d1" size="4">
                Métodos Numéricos &bull; Sistemas de ecuaciones
            </font></p>
            <p style="font-style: oblique;"><font color="#1957d1" size="3">
                Mario Cueva &bull; Andrés Merino &bull; Período 2026-1
            </font></p>
        </td>
    </tr>
</table>

---
## <font color='264CC7'> Introducción </font>

Este cuaderno de trabajo permite practicar los conceptos del **Resumen no. 3: Sistemas de ecuaciones**.

Al finalizar, se espera que puedas:

- Evaluar sistemas no lineales y construir su matriz jacobiana.
- Aplicar punto fijo y Newton-Raphson a sistemas no lineales.
- Relacionar la linealización con el paso de Newton.
- Resolver sistemas lineales mediante eliminación de Gauss y pivoteo parcial.
- Interpretar el número de condición y practicar factorización LU e inversión de matrices.

Los paquetes necesarios son:

In [ ]:
# Paquetes necesarios
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## <font color='264CC7'> Funciones auxiliares </font>

Usaremos normas para medir residuos y cambios entre aproximaciones vectoriales.

In [ ]:
def norma_vector(vector):
    return np.linalg.norm(np.asarray(vector, dtype=float), ord=np.inf)


def error_vectorial(x_actual, x_anterior):
    if x_actual is None or x_anterior is None:
        return None
    return norma_vector(np.asarray(x_actual) - np.asarray(x_anterior))


def norma_residuo(funcion, x):
    if x is None:
        return None
    return norma_vector(funcion(np.asarray(x, dtype=float)))


def formato_vector(vector, decimales=6):
    if vector is None:
        return None
    return np.array2string(np.asarray(vector, dtype=float), precision=decimales, suppress_small=True)

---
## <font color='264CC7'> 1. Sistemas no lineales y matriz jacobiana </font>

Para un sistema no lineal escribimos

$$
F(x)=\begin{pmatrix}f_1(x)\\f_2(x)\\\vdots\\f_n(x)\end{pmatrix}=0.
$$

La matriz jacobiana reúne las derivadas parciales de las ecuaciones del sistema.

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 1:</strong><br>
Completa la matriz jacobiana del sistema formado por la circunferencia <code>x**2+y**2=4</code> y la recta <code>x=y</code>. Evalúa el sistema y el jacobiano en <code>(1,1)</code>.
</div>

In [ ]:
def F_circulo(vector):
    x, y = vector
    return np.array([
        x**2 + y**2 - 4,
        x - y,
    ], dtype=float)


def J_circulo(vector):
    x, y = vector
    # TODO: reemplaza las entradas del jacobiano.
    return np.array([
        [None, None],
        [None, None],
    ])


x0_circulo = np.array([1.0, 1.0])
print('F(1,1) =', F_circulo(x0_circulo))
# J_circulo(x0_circulo)

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 2:</strong><br>
Visualiza las curvas <code>f1(x,y)=0</code> y <code>f2(x,y)=0</code>. Identifica en la gráfica las soluciones del sistema y comprueba que su residuo sea pequeño.
</div>

In [ ]:
xs = np.linspace(-2.5, 2.5, 300)
ys = np.linspace(-2.5, 2.5, 300)
X, Y = np.meshgrid(xs, ys)

f1 = X**2 + Y**2 - 4
f2 = X - Y

plt.contour(X, Y, f1, levels=[0], colors=['#264CC7'])
plt.contour(X, Y, f2, levels=[0], colors=['#D1495B'])
plt.axis('equal')
plt.grid(alpha=0.25)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Intersecciones de las ecuaciones del sistema')
plt.show()

# TODO: escribe las dos soluciones observadas y calcula norma_residuo(F_circulo, solucion).
# solucion_1 = np.array([..., ...])
# solucion_2 = np.array([..., ...])

---
## <font color='264CC7'> 2. Punto fijo para sistemas </font>

Una transformación de punto fijo tiene la forma

$$
x^{(k+1)}=G\bigl(x^{(k)}\bigr).
$$

La elección de `G` influye en la convergencia. Una condición suficiente involucra una cota menor que `1` para una norma de su jacobiano.

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 3:</strong><br>
Completa el método de punto fijo para sistemas. Aplícalo a la transformación del sistema <code>3x-y-1=0</code> y <code>4y-x**2-1=0</code>.
</div>

In [ ]:
def punto_fijo_sistemas(G, x0, tolerancia=1e-6, max_iter=100):
    filas = []
    x_actual = np.asarray(x0, dtype=float)

    for i in range(1, max_iter + 1):
        # TODO: calcula la nueva aproximación con G.
        x_nuevo = None
        err = error_vectorial(x_nuevo, x_actual)

        filas.append({
            'iteracion': i,
            'x_anterior': formato_vector(x_actual),
            'x': formato_vector(x_nuevo),
            'error_aprox': err,
        })

        # TODO: detén la iteración cuando err sea menor que la tolerancia.

        x_actual = x_nuevo

    return pd.DataFrame(filas)


def G_sistema(vector):
    x, y = vector
    return np.array([(1 + y) / 3, (1 + x**2) / 4], dtype=float)


# tabla_pf = punto_fijo_sistemas(G_sistema, x0=[0, 0], tolerancia=1e-8, max_iter=50)
# tabla_pf.tail()

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 4:</strong><br>
Estima la norma infinito del jacobiano de <code>G_sistema</code> en varios puntos de una región cercana a la iteración. ¿Qué sugiere sobre la convergencia?
</div>

In [ ]:
def J_G_sistema(vector):
    x, y = vector
    return np.array([
        [0, 1 / 3],
        [x / 2, 0],
    ], dtype=float)


puntos_prueba = [np.array([0.0, 0.0]), np.array([0.4, 0.3]), np.array([0.5, 0.35])]

# TODO: calcula np.linalg.norm(J_G_sistema(punto), ord=np.inf) para cada punto.
# for punto in puntos_prueba:
#     print(formato_vector(punto), '->', ...)

---
## <font color='264CC7'> 3. Newton-Raphson para sistemas </font>

Newton-Raphson resuelve en cada iteración un sistema lineal para la corrección `s`:

$$
J_F\bigl(x^{(k)}\bigr)s^{(k)}=-F\bigl(x^{(k)}\bigr),
\qquad
x^{(k+1)}=x^{(k)}+s^{(k)}.
$$

En el código es preferible resolver ese sistema con `np.linalg.solve` en lugar de calcular la inversa del jacobiano.

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 5:</strong><br>
Completa Newton-Raphson para sistemas y aplícalo al sistema del círculo desde <code>x0=[1,1]</code>.
</div>

In [ ]:
def newton_sistemas(F, J, x0, tolerancia=1e-6, max_iter=100):
    filas = []
    x_actual = np.asarray(x0, dtype=float)

    for i in range(1, max_iter + 1):
        Fx = F(x_actual)
        Jx = J(x_actual)

        # TODO: resuelve Jx @ paso = -Fx.
        paso = None
        x_nuevo = x_actual + paso if paso is not None else None
        err = error_vectorial(x_nuevo, x_actual)

        filas.append({
            'iteracion': i,
            'x_anterior': formato_vector(x_actual),
            'x': formato_vector(x_nuevo),
            'norma_residuo': norma_residuo(F, x_nuevo),
            'norma_paso': norma_vector(paso) if paso is not None else None,
            'error_aprox': err,
        })

        # TODO: usa err o la norma del residuo como criterio de parada.

        x_actual = x_nuevo

    return pd.DataFrame(filas)


# tabla_newton = newton_sistemas(F_circulo, J_circulo, x0=[1, 1], tolerancia=1e-10, max_iter=20)
# tabla_newton

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 6:</strong><br>
Calcula manualmente la primera corrección de Newton en <code>(1,1)</code>. Compara el resultado con la linealización del sistema alrededor de ese punto.
</div>

In [ ]:
x_linealizacion = np.array([1.0, 1.0])
F0 = F_circulo(x_linealizacion)

# TODO: después de completar J_circulo, calcula la primera corrección.
# J0 = J_circulo(x_linealizacion)
# s0 = np.linalg.solve(J0, -F0)
# x1 = x_linealizacion + s0
# print('corrección =', s0)
# print('primera aproximación =', x1)

La linealización de un sistema cerca de `x0` se escribe como

$$
F(x)\approx F(x_0)+J_F(x_0)(x-x_0).
$$

Igualar esta aproximación a cero produce el mismo sistema lineal que resuelve Newton para obtener la corrección.

---
## <font color='264CC7'> 4. Eliminación de Gauss y pivoteo parcial </font>

Un sistema lineal se representa como `A @ x = b`. La eliminación de Gauss transforma la matriz aumentada en un sistema triangular superior y luego aplica sustitución regresiva.

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 7:</strong><br>
Completa la sustitución regresiva y úsala con el sistema triangular que aparece tras una eliminación de Gauss.
</div>

In [ ]:
def sustitucion_regresiva(U, y):
    U = np.asarray(U, dtype=float)
    y = np.asarray(y, dtype=float)
    x = np.zeros_like(y)

    for i in range(len(y) - 1, -1, -1):
        if U[i, i] == 0:
            raise ZeroDivisionError('Hay un pivote nulo en la matriz triangular.')
        # TODO: despeja x[i] usando los valores ya calculados.
        x[i] = None

    return x


U_ejemplo = np.array([
    [2, 1, -1],
    [0, 0.5, 0.5],
    [0, 0, -1],
], dtype=float)
y_ejemplo = np.array([8, 1, 1], dtype=float)

# sustitucion_regresiva(U_ejemplo, y_ejemplo)

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 8:</strong><br>
Completa la eliminación de Gauss. Activa el pivoteo parcial para elegir el mayor pivote disponible en valor absoluto.
</div>

In [ ]:
def eliminacion_gauss(A, b, pivoteo=True):
    A = np.asarray(A, dtype=float).copy()
    b = np.asarray(b, dtype=float).copy()
    n = len(b)

    for k in range(n - 1):
        if pivoteo:
            # TODO: localiza la fila con el mayor abs(A[fila, k]) desde k hasta n-1.
            fila_pivote = None
            # TODO: intercambia filas en A y b cuando fila_pivote sea distinta de k.

        if A[k, k] == 0:
            raise ZeroDivisionError('No existe un pivote utilizable en esta columna.')

        for i in range(k + 1, n):
            # TODO: calcula el multiplicador y elimina bajo el pivote.
            multiplicador = None
            # A[i, k:] = ...
            # b[i] = ...

    return A, b


A_gauss = np.array([
    [2, 1, -1],
    [-3, -1, 2],
    [-2, 1, 2],
], dtype=float)
b_gauss = np.array([8, -11, -3], dtype=float)

# U_gauss, y_gauss = eliminacion_gauss(A_gauss, b_gauss, pivoteo=True)
# sustitucion_regresiva(U_gauss, y_gauss)

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 9:</strong><br>
Prueba una matriz con primer pivote pequeño. Compara los sistemas triangulares obtenidos con y sin pivoteo.
</div>

In [ ]:
A_pivote = np.array([
    [0.0001, 1],
    [1, 1],
], dtype=float)
b_pivote = np.array([1.0001, 2], dtype=float)

# TODO: ejecuta eliminacion_gauss con pivoteo=False y pivoteo=True.
# U_sin, y_sin = eliminacion_gauss(A_pivote, b_pivote, pivoteo=False)
# U_con, y_con = eliminacion_gauss(A_pivote, b_pivote, pivoteo=True)
# print('Sin pivoteo:\n', U_sin, '\n', y_sin)
# print('Con pivoteo:\n', U_con, '\n', y_con)

---
## <font color='264CC7'> 5. Condicionamiento de sistemas lineales </font>

El número de condición asociado a una norma se puede escribir como

$$
\kappa(A)=\|A\|\|A^{-1}\|.
$$

Una matriz con número de condición grande puede amplificar perturbaciones pequeñas en los datos.

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 10:</strong><br>
Resuelve dos sistemas con la misma matriz casi singular y términos independientes ligeramente distintos. Calcula <code>np.linalg.cond</code> y explica el cambio en la solución.
</div>

In [ ]:
A_cond = np.array([
    [1, 1],
    [1, 1.0001],
], dtype=float)
b_cond = np.array([2, 2.0001], dtype=float)
b_perturbado = np.array([2, 2.0002], dtype=float)

x_cond = np.linalg.solve(A_cond, b_cond)
x_perturbado = np.linalg.solve(A_cond, b_perturbado)

print('solución original =', x_cond)
print('solución perturbada =', x_perturbado)
print('cambio en b =', b_perturbado - b_cond)

# TODO: calcula el número de condición con la norma infinito.
# condicion = np.linalg.cond(A_cond, p=np.inf)
# condicion

---
## <font color='264CC7'> 6. Factorización LU </font>

Si no se requieren intercambios de filas, la eliminación de Gauss puede organizarse como

$$
A=LU,
$$

donde `L` guarda los multiplicadores y `U` es triangular superior.

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 11:</strong><br>
Completa la factorización LU sin pivoteo y verifica que <code>L @ U</code> reconstruya la matriz original.
</div>

In [ ]:
def factorizacion_lu(A):
    U = np.asarray(A, dtype=float).copy()
    n = U.shape[0]
    L = np.eye(n)

    for k in range(n - 1):
        if U[k, k] == 0:
            raise ZeroDivisionError('Esta versión LU necesita pivotes no nulos.')

        for i in range(k + 1, n):
            # TODO: guarda el multiplicador en L y elimina la fila de U.
            multiplicador = None
            # L[i, k] = ...
            # U[i, k:] = ...

    return L, U


A_lu = np.array([
    [2, 1, 1],
    [4, -6, 0],
    [-2, 7, 2],
], dtype=float)

# L, U = factorizacion_lu(A_lu)
# print('L =\n', L)
# print('U =\n', U)
# print('error de reconstrucción =', np.linalg.norm(A_lu - L @ U))

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 12:</strong><br>
Completa la sustitución progresiva. Luego resuelve <code>A_lu @ x = b_lu</code> usando primero <code>L @ y = b_lu</code> y después <code>U @ x = y</code>.
</div>

In [ ]:
def sustitucion_progresiva(L, b):
    L = np.asarray(L, dtype=float)
    b = np.asarray(b, dtype=float)
    y = np.zeros_like(b)

    for i in range(len(b)):
        if L[i, i] == 0:
            raise ZeroDivisionError('Hay un pivote nulo en la matriz triangular.')
        # TODO: despeja y[i] usando las componentes anteriores.
        y[i] = None

    return y


b_lu = np.array([5, -2, 9], dtype=float)

# y_lu = sustitucion_progresiva(L, b_lu)
# x_lu = sustitucion_regresiva(U, y_lu)
# print('solución con LU =', x_lu)
# print('solución de referencia =', np.linalg.solve(A_lu, b_lu))

---
## <font color='264CC7'> 7. Inversión de matrices </font>

Las columnas de `A^{-1}` son las soluciones de

$$
Ax_j=e_j,
$$

donde `e_j` es una columna de la identidad. Para resolver un solo sistema suele ser mejor resolverlo directamente o reutilizar una factorización.

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 13:</strong><br>
Construye la inversa por columnas resolviendo un sistema por cada columna de la identidad. Verifica que el producto con <code>A_inversa</code> sea la identidad.
</div>

In [ ]:
A_inversa = np.array([
    [2, 1],
    [5, 3],
], dtype=float)
I = np.eye(A_inversa.shape[0])

# TODO: resuelve A_inversa @ columna = I[:, j] para cada j.
columnas_inversa = []
# for j in range(I.shape[1]):
#     columna = ...
#     columnas_inversa.append(columna)

# inversa_por_columnas = np.column_stack(columnas_inversa)
# print(inversa_por_columnas)
# print(A_inversa @ inversa_por_columnas)
# print(np.linalg.inv(A_inversa))

---
## <font color='264CC7'> Comparación de ideas </font>

Completa la tabla conceptual antes de cerrar el cuaderno.

| Tema | Objeto principal | Cálculo que se repite | Riesgo a vigilar |
|---|---|---|---|
| Punto fijo para sistemas | Transformación `G` | Evaluar `G(x)` | La transformación puede divergir |
| Newton para sistemas | Sistema no lineal `F(x)=0` | Resolver con el jacobiano | Jacobiano singular o inicio lejano |
| Gauss | Sistema lineal `A @ x = b` | Eliminación y sustitución | Pivotes pequeños |
| Condicionamiento | Matriz `A` | Medir sensibilidad | Errores amplificados |
| LU | Matriz reutilizada | Dos sustituciones triangulares | Necesidad de permutaciones |

<div style="background-color: #edf1f8; border-color: #264CC7; border-left: 5px solid #264CC7; padding: 0.5em;">
<strong>Ejercicio 14:</strong><br>
Cuando hayas completado los métodos, compara el residuo de Newton para el sistema no lineal y el residuo de Gauss o LU para un sistema lineal.
</div>

In [ ]:
def residuo_lineal(A, x, b):
    return norma_vector(np.asarray(A) @ np.asarray(x) - np.asarray(b))


# TODO: usa tus resultados previos.
# print('residuo del último Newton =', tabla_newton.iloc[-1]['norma_residuo'])
# print('residuo del sistema resuelto por LU =', residuo_lineal(A_lu, x_lu, b_lu))

---
## <font color='264CC7'> Preguntas de cierre </font>

Responde en tus propias palabras:

1. ¿Qué información aporta el jacobiano en Newton-Raphson para sistemas?
2. ¿Por qué una transformación de punto fijo puede fallar aunque sea equivalente al sistema original?
3. ¿Qué problema intenta reducir el pivoteo parcial durante la eliminación de Gauss?
4. ¿Por qué el número de condición describe al problema y no solo al algoritmo?
5. ¿Cuándo conviene reutilizar una factorización LU?
6. ¿Por qué no es necesario calcular una inversa para resolver un único sistema lineal?

In [ ]:
# Espacio para pruebas adicionales